# MOHIM raw + mean-centered motif dataset and stem-wise threshold diagnostics

음원을 source 단위로 한 번 분리해 dataset stem과 앞 30초의 모든 4마디 후보 진단 결과를 함께 저장합니다.

## 0. Drive와 mean-centering 브랜치 준비

In [21]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import subprocess

REPOSITORY = 'https://github.com/youhan200203/MOHIM.git'
BRANCH = 'mean-centering'
REPO_DIR = Path('/content/MOHIM')
if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPOSITORY, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'switch', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print('repository:', REPO_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
repository: /content/MOHIM


In [2]:
%pip install -q -r requirements.txt
%pip install -q --no-deps "beat-this @ git+https://github.com/CPJKU/beat_this.git"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 14.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 36.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 916.9/916.9 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 379.0/379.0 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/10

## 1. 실험 경로와 범위

In [52]:
MAX_SONGS = 3000
DATA_SOURCE = 'billboard'  # 'songs' 또는 'billboard'
FORCE_REPROCESS = True
DEVICE = 'cuda'
DEMUCS_BATCH_SIZE = 8
MOTIF_WORKERS = 12
DEMUCS_SHIFTS = 0
AUDIO_FORMAT = 'flac'
MOTIF_BARS = 4
MOTIF_SEARCH_SECONDS = 30.0
MIN_ACTIVE_RATIO = 0.65
MAX_ONSET_CHROMA_DIFFERENCE = 0.40
MIN_ONSET_SIMILARITY = 0.54
MIN_PITCH_CLASS_SPAN = 0.25
MIN_MEAN_CENTERED_SIMILARITY = 0.25
MIN_ONSET_VARIATION = 0.23
SIMILARITY_MODE = 'raw_gate_mean_centered_rank_v1'
MIN_NEXT_BAR_SCORE_GAIN = 0.05

BAR_ALIGNMENT_SCORE_KEYS = (
    'onset_similarity',
    'chroma_similarity',
    'mean_centered_onset_similarity',
    'mean_centered_chroma_similarity',
)

SONGS_DIR = Path('/content/drive/MyDrive/MOHIM/songs')
BILLBOARD_DATASET_DIR = Path('/content/drive/MyDrive/MOHIM/billboard_pop_dataset')
MOTIF_DATASET_DIR = Path('/content/drive/MyDrive/MOHIM/motif_dataset_mean_centered')
MOTIF_VERSION_PATH = MOTIF_DATASET_DIR / '.mohim_motif_version'
SONGS_STEM_DIR = Path('/content/drive/MyDrive/MOHIM/songs_stem_dataset')
DIAGNOSTIC_DIR = Path('/content/drive/MyDrive/MOHIM/motif_stem_diagnostics_mean_centered')
BEAT_CHECKPOINT = Path('/content/checkpoints/beat_this_final0.ckpt')
AUDIO_EXTENSIONS = {'.aac', '.flac', '.m4a', '.mp3', '.ogg', '.wav', '.webm'}

assert DATA_SOURCE in {'songs', 'billboard'}
MOTIF_DATASET_DIR.mkdir(parents=True, exist_ok=True)
SONGS_STEM_DIR.mkdir(parents=True, exist_ok=True)
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)

## 1-1. `songs` 폴더 입력

`DATA_SOURCE = 'songs'`일 때만 실행되며 파일명을 곡 제목으로 사용합니다.

In [37]:
if DATA_SOURCE == 'songs':
    song_paths = sorted(
        path for path in SONGS_DIR.iterdir()
        if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS
    )[:MAX_SONGS]
    songs = [
        {
            'track_id': path.stem,
            'artist': '',
            'title': path.stem,
            'audio_path': path,
        }
        for path in song_paths
    ]

## 1-2. Billboard dataset 입력

`DATA_SOURCE = 'billboard'`일 때 `billboard_pop_dataset/tracks.json`과 `audio` 폴더에서 실제 음원이 있는 곡을 최대 10개 가져옵니다.

In [38]:
if DATA_SOURCE == 'billboard':
    from mohim.dataset import index_audio_files, load_local_tracks, resolve_audio_path

    tracks_json = BILLBOARD_DATASET_DIR / 'tracks_popular.json'
    audio_dir = BILLBOARD_DATASET_DIR / 'audio'
    assert tracks_json.is_file(), f'tracks.json이 없습니다: {tracks_json}'
    assert audio_dir.is_dir(), f'음원 폴더가 없습니다: {audio_dir}'

    tracks = load_local_tracks(tracks_json, require_lyrics=True)
    audio_index = index_audio_files(audio_dir)
    songs = []
    for track in tracks:
        audio_path = resolve_audio_path(track, audio_index)
        if audio_path is None:
            print(f'[skip] 음원 없음: {track.artist} - {track.title}')
            continue
        songs.append({
            'track_id': track.track_id,
            'artist': track.artist,
            'title': track.title,
            'audio_path': audio_path,
            'track': track,
        })
        if len(songs) >= MAX_SONGS:
            break
    song_paths = [song['audio_path'] for song in songs]

## 1-3. 선택한 입력 확인

In [43]:
songs = songs[10:]
song_paths = [song['audio_path'] for song in songs]

In [48]:
assert songs, f'사용 가능한 음원이 없습니다: {DATA_SOURCE}'
assert len(songs) == len(song_paths)
print(f'data source: {DATA_SOURCE}')
print('songs:', len(songs))
for song in songs:
    label = f"{song['artist']} - {song['title']}" if song['artist'] else song['title']
    print('-', label)

data source: billboard
songs: 55
- Doja Cat - Agora Hills
- Doja Cat - Need To Know
- Jack Harlow - Lovin On Me
- Justin Bieber - Yukon
- SHAED - Trampoline
- Ed Sheeran - Shape Of You
- Olivia Rodrigo - Good 4 U
- Post Malone - Better Now
- Shawn Mendes - Stitches
- Alessia Cara - Scars To Your Beautiful
- Jonas Brothers - Sucker
- Lizzo - Truth Hurts
- LMFAO Featuring Lauren Bennett & GoonRock - Party Rock Anthem
- Panic! At The Disco - High Hopes
- Kehlani - Folded
- Dua Lipa - Break My Heart
- Ellie Goulding - Lights
- Justin Bieber - Daisies
- Khalid - Talk
- Olivia Dean - So Easy (To Fall In Love)
- Sabrina Carpenter - Taste
- SZA - Kill Bill
- The All-American Rejects - Gives You Hell
- DNCE - Cake By The Ocean
- Camila Cabello - Never Be The Same
- Charlie Puth - Attention
- Trevor Daniel - Falling
- Benson Boone - Sorry I'm Here For Someone Else
- Ed Sheeran - Bad Habits
- SZA - Snooze
- Tate McRae - You Broke Me First.
- Kevin Rudolf Featuring Lil Wayne - Let It Rock
- The Sc

## 2. Demucs와 motif scorer 준비

In [53]:
import urllib.request
from mohim.motif import MotifConfig, MotifExtractor, create_beat_tracker, onset_variation
from mohim.dataset import DatasetBuilder
from mohim.separator import StemSeparator

BEAT_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if not BEAT_CHECKPOINT.is_file():
    urllib.request.urlretrieve(
        'https://cloud.cp.jku.at/public.php/dav/files/7ik4RrBKTS273gp/final0.ckpt', BEAT_CHECKPOINT
    )
separator = StemSeparator(
    device=DEVICE, model_name='htdemucs_6s', shifts=DEMUCS_SHIFTS
)
beat_tracker = create_beat_tracker(BEAT_CHECKPOINT, device=DEVICE)
motif_scorer = MotifExtractor(
    beat_tracker,
    MotifConfig(
        bars=MOTIF_BARS,
        search_seconds=MOTIF_SEARCH_SECONDS,
        min_presence=MIN_ACTIVE_RATIO,
        max_similarity_difference=MAX_ONSET_CHROMA_DIFFERENCE,
        onset_threshold=MIN_ONSET_SIMILARITY,
        pitch_class_span_threshold=MIN_PITCH_CLASS_SPAN,
        mean_centered_similarity_threshold=MIN_MEAN_CENTERED_SIMILARITY,
        onset_variation_threshold=MIN_ONSET_VARIATION,
    ),
)

dataset_builder = (
    DatasetBuilder(
        audio_dir=BILLBOARD_DATASET_DIR / 'audio',
        output_dir=MOTIF_DATASET_DIR,
        separator=separator,
        motif_extractor=motif_scorer,
        audio_format=AUDIO_FORMAT,
        resume=not FORCE_REPROCESS,
    )
    if DATA_SOURCE == 'billboard' else None
)
print('separator and motif scorer ready')

separator and motif scorer ready


## 3. 모든 stem × start-downbeat 후보 계산

`DEMUCS_BATCH_SIZE`개 곡을 한 GPU batch로 분리하고, 해당 batch의 곡 × stem 점수 계산은 `MOTIF_WORKERS`개 CPU thread로 병렬 실행합니다. Raw onset/chroma는 1차 후보 필터에 사용하고, mean-centered onset/chroma는 stem별 최초 후보의 최종 순위와 0.28 하한 검사에 사용합니다. Onset 하한에 미달한 후보는 pitch-class span 계산을 생략하며, `candidate_*.flac`은 active ratio, raw onset/chroma 차이, raw onset, pitch-class span 조건까지 통과한 구간만 저장합니다.

In [ ]:
from collections import Counter
import json
import librosa
import numpy as np
import pandas as pd
import re
import shutil
from mohim.separator import save_audio

def average_next_bar_score_gain(current, next_candidate):
    """
    현재 후보보다 다음 마디 후보의 4개 점수가
    평균적으로 얼마나 좋아졌는지 계산.
    """
    gains = [
        next_candidate[key] - current[key]
        for key in BAR_ALIGNMENT_SCORE_KEYS
    ]

    return float(np.mean(gains))

def safe_folder_name(value):
    cleaned = re.sub(r'[\\/:*?"<>|\x00-\x1f]', '_', value)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip(' .')
    return cleaned[:120] or 'untitled'

def song_display_title(song):
    return f"{song['artist']} - {song['title']}" if song['artist'] else song['title']

diagnostic_base_name_counts = Counter(
    safe_folder_name(song_display_title(song)) for song in songs
)

def diagnostic_song_id(song):
    base_name = safe_folder_name(song_display_title(song))
    if diagnostic_base_name_counts[base_name] == 1:
        return base_name
    short_track_id = safe_folder_name(str(song['track_id']))[:8]
    return f'{base_name[:109]} [{short_track_id}]'

def song_stem_metadata_path(song):
    display_title = song_display_title(song)
    return SONGS_STEM_DIR / safe_folder_name(display_title) / 'metadata.json'

def song_stems_complete(song, audio_path):
    metadata_path = song_stem_metadata_path(song)
    if not metadata_path.is_file():
        return False
    try:
        metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
        same_audio = Path(metadata.get('source_audio', '')).resolve() == Path(audio_path).resolve()
        stem_files = metadata.get('stem_files')
        return (
            same_audio
            and isinstance(stem_files, dict)
            and stem_files
            and all((metadata_path.parent / filename).is_file() for filename in stem_files.values())
        )
    except (OSError, ValueError, TypeError):
        return False

def save_song_stems(song, audio_path, stems, sample_rate):
    metadata_path = song_stem_metadata_path(song)
    metadata_path.parent.mkdir(parents=True, exist_ok=True)
    stem_files = {}
    for stem_name, stem_audio in stems.items():
        if not stem_name.replace('_', '').isalnum():
            raise ValueError(f'Unsafe separator stem name: {stem_name!r}')
        filename = f'{stem_name}.{AUDIO_FORMAT}'
        save_audio(metadata_path.parent / filename, stem_audio, sample_rate, audio_format=AUDIO_FORMAT)
        stem_files[stem_name] = filename
    metadata = {
        'track_id': song['track_id'], 'artist': song['artist'], 'title': song['title'],
        'source_audio': str(audio_path), 'sample_rate': sample_rate, 'stem_files': stem_files,
    }
    metadata_path.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    return metadata_path.parent

def internal_candidate_features(audio_path):
    audio, sample_rate = librosa.load(audio_path, sr=None, mono=True)
    hop_length = 512
    chroma = librosa.feature.chroma_cens(
        y=audio, sr=sample_rate, hop_length=hop_length
    )
    chroma_sum = chroma.sum(axis=0, keepdims=True)
    valid_chroma = chroma_sum.ravel() > 1e-8
    chroma_norm = np.divide(
        chroma, chroma_sum, out=np.zeros_like(chroma), where=chroma_sum > 1e-8
    )
    valid_pairs = valid_chroma[:-1] & valid_chroma[1:]
    if np.any(valid_pairs):
        frame_flux = 0.5 * np.sum(np.abs(np.diff(chroma_norm, axis=1)), axis=0)
        chroma_flux = float(np.mean(frame_flux[valid_pairs]))
    else:
        chroma_flux = 0.0
    return {'chroma_flux': chroma_flux}

def candidate_passes_export_filters(row):
    return (
        row['active_ratio'] >= MIN_ACTIVE_RATIO
        and abs(row['onset_similarity'] - row['chroma_similarity']) <= MAX_ONSET_CHROMA_DIFFERENCE
        and row['onset_similarity'] >= MIN_ONSET_SIMILARITY
        and row['pitch_class_span'] is not None
        and row['pitch_class_span'] >= MIN_PITCH_CLASS_SPAN
    )

motif_result_counts = {'successed': 0, 'failed': 0}

def print_motif_result(title, succeeded, detail=None):
    result_key = 'successed' if succeeded else 'failed'
    motif_result_counts[result_key] += 1
    label = 'selection' if succeeded else 'skipped'
    counts = (
        f"successed: {motif_result_counts['successed']} / "
        f"failed: {motif_result_counts['failed']}"
    )
    detail_suffix = f' - {detail}' if detail else ''
    print(f'  [{label}] {title} ({counts}){detail_suffix}')

MIN_NEXT_BAR_SCORE_GAIN = 0.05

BAR_ALIGNMENT_SCORE_KEYS = (
    'onset_similarity',
    'chroma_similarity',
    'mean_centered_onset_similarity',
    'mean_centered_chroma_similarity',
)


def average_next_bar_score_gain(current, next_candidate):
    """
    현재 후보와 다음 마디 후보의 4개 점수 차이(next - current)를
    평균내서 다음 마디로 이동할지 판단.
    """
    return float(np.mean([
        next_candidate[key] - current[key]
        for key in BAR_ALIGNMENT_SCORE_KEYS
    ]))


def save_earliest_valid_selection(metadata_path, metadata):
    # 이전 selection 파일 제거
    for stale_name in (
        'motif_selections.json',
        'selected_earliest.flac',
        'selected_highest_similarity.flac',
    ):
        stale_path = metadata_path.parent / stale_name
        if stale_path.is_file():
            stale_path.unlink()

    # =========================================================
    # 1. 1차 유효성 필터
    #
    # 여기서는:
    #   - active_ratio
    #   - onset_similarity
    #   - onset/chroma difference
    #   - pitch_class_span
    #
    # 만 검사.
    #
    # mean-centered / onset_variation은 아직 검사하지 않음.
    # =========================================================
    eligible = [
        row
        for row in metadata['candidates']
        if candidate_passes_export_filters(row)
    ]

    if not eligible:
        skipped_metadata = {
            'song_id': metadata['song_id'],
            'title': metadata['title'],
            'source_audio': metadata['source_audio'],
            'status': 'skipped',
            'reason': 'no_candidate_passed_export_filters',

            'min_active_ratio': MIN_ACTIVE_RATIO,
            'max_onset_chroma_difference': MAX_ONSET_CHROMA_DIFFERENCE,
            'onset_threshold': MIN_ONSET_SIMILARITY,
            'pitch_class_span_threshold': MIN_PITCH_CLASS_SPAN,
            'mean_centered_similarity_threshold': MIN_MEAN_CENTERED_SIMILARITY,
            'onset_variation_threshold': MIN_ONSET_VARIATION,
            'next_bar_score_gain_threshold': MIN_NEXT_BAR_SCORE_GAIN,

            'similarity_mode': SIMILARITY_MODE,
            'candidate_count': len(metadata['candidates']),
            'eligible_candidate_count': 0,

            'alignment_decisions': {},
            'selected_by_stem_before_final_check': {},
            'final_validation': {},
            'selections': {},
        }

        (metadata_path.parent / 'motif_selections.json').write_text(
            json.dumps(
                skipped_metadata,
                ensure_ascii=False,
                indent=2,
            ),
            encoding='utf-8',
        )

        print_motif_result(
            metadata['title'],
            False,
            '1차 조건 통과 후보 없음',
        )

        return skipped_metadata

    # =========================================================
    # 2. stem별로 1차 유효 후보 묶기
    # =========================================================
    candidates_by_stem = {}

    for row in eligible:
        candidates_by_stem.setdefault(
            row['stem_name'],
            [],
        ).append(row)

    # 각 stem 내부는 시간순 정렬
    for stem_name in candidates_by_stem:
        candidates_by_stem[stem_name].sort(
            key=lambda row: row['start_sec']
        )

    # =========================================================
    # 3. stem 내부 시작점 결정
    #
    # 가장 이른 1차 유효 후보부터 시작.
    #
    # current → next의 4개 점수 평균 개선량이
    # 0.05 이상이면 다음 마디로 이동.
    #
    # 이 단계에서는 최종 threshold를 검사하지 않음.
    # =========================================================
    selected_by_stem_before_final_check = {}
    alignment_decisions = {}

    for stem_name, stem_rows in candidates_by_stem.items():
        current_pos = 0
        current = stem_rows[current_pos]

        decisions = []

        while current_pos + 1 < len(stem_rows):
            next_pos = current_pos + 1
            next_candidate = stem_rows[next_pos]

            score_differences = {
                key: float(
                    next_candidate[key] - current[key]
                )
                for key in BAR_ALIGNMENT_SCORE_KEYS
            }

            average_gain = float(np.mean(
                list(score_differences.values())
            ))

            decision = {
                'current_candidate_index': current['candidate_index'],
                'current_start_sec': current['start_sec'],

                'next_candidate_index': next_candidate['candidate_index'],
                'next_start_sec': next_candidate['start_sec'],

                'score_differences': score_differences,
                'average_score_gain': average_gain,
                'threshold': MIN_NEXT_BAR_SCORE_GAIN,
            }

            # ---------------------------------------------
            # 다음 마디가 평균적으로 0.05 이상 더 좋으면 이동
            # ---------------------------------------------
            if average_gain >= MIN_NEXT_BAR_SCORE_GAIN:
                decision['action'] = 'move_to_next_bar'
                decisions.append(decision)

                current_pos = next_pos
                current = next_candidate

                # 이동했으므로 새 current와 그 다음 후보를 다시 비교
                continue

            # ---------------------------------------------
            # 충분한 차이가 아니면 현재 시작점 유지
            # ---------------------------------------------
            decision['action'] = 'keep_current'
            decisions.append(decision)

            break

        selected_by_stem_before_final_check[stem_name] = dict(current)

        alignment_decisions[stem_name] = {
            'initial_candidate_index': stem_rows[0]['candidate_index'],
            'initial_start_sec': stem_rows[0]['start_sec'],

            'selected_candidate_index': current['candidate_index'],
            'selected_start_sec': current['start_sec'],

            'decisions': decisions,
        }

    # =========================================================
    # 4. 이제 stem별로 선택된 후보에 대해서만 최종 점검
    #
    #   - mean-centered onset similarity
    #   - mean-centered chroma similarity
    #   - onset variation
    #
    # 여기서 실패하면 그 stem은 최종 후보에서 탈락.
    # 다른 candidate로 fallback하지 않음.
    # =========================================================
    final_valid_by_stem = {}
    final_validation = {}

    for stem_name, selected_row in (
        selected_by_stem_before_final_check.items()
    ):
        row = dict(selected_row)

        validation = {
            'candidate_index': row['candidate_index'],
            'start_sec': row['start_sec'],

            'mean_centered_onset_similarity':
                row['mean_centered_onset_similarity'],

            'mean_centered_chroma_similarity':
                row['mean_centered_chroma_similarity'],
        }

        # ---------------------------------------------
        # 4-1. mean-centered similarity 최종 검사
        # ---------------------------------------------
        mean_centered_onset_pass = (
            row['mean_centered_onset_similarity']
            >= MIN_MEAN_CENTERED_SIMILARITY
        )

        mean_centered_chroma_pass = (
            row['mean_centered_chroma_similarity']
            >= MIN_MEAN_CENTERED_SIMILARITY
        )

        validation['mean_centered_onset_pass'] = (
            mean_centered_onset_pass
        )

        validation['mean_centered_chroma_pass'] = (
            mean_centered_chroma_pass
        )

        if not (
            mean_centered_onset_pass
            and mean_centered_chroma_pass
        ):
            validation['passed'] = False
            validation['reason'] = (
                'mean_centered_similarity_below_threshold'
            )

            final_validation[stem_name] = validation
            continue

        # ---------------------------------------------
        # 4-2. onset variation 최종 검사
        # ---------------------------------------------
        candidate_path = (
            metadata_path.parent
            / row['candidate_file']
        )

        candidate_audio, candidate_sample_rate = librosa.load(
            candidate_path,
            sr=None,
            mono=False,
        )

        variation = onset_variation(
            candidate_audio,
            candidate_sample_rate,
        )

        row['onset_variation'] = variation

        validation['onset_variation'] = variation
        validation['onset_variation_pass'] = (
            variation >= MIN_ONSET_VARIATION
        )

        if variation < MIN_ONSET_VARIATION:
            validation['passed'] = False
            validation['reason'] = (
                'onset_variation_below_threshold'
            )

            final_validation[stem_name] = validation
            continue

        # ---------------------------------------------
        # 최종 통과
        # ---------------------------------------------
        validation['passed'] = True
        validation['reason'] = None

        final_validation[stem_name] = validation
        final_valid_by_stem[stem_name] = row

    # =========================================================
    # 5. 모든 stem이 최종 검사에서 탈락한 경우
    # =========================================================
    if not final_valid_by_stem:
        skipped_metadata = {
            'song_id': metadata['song_id'],
            'title': metadata['title'],
            'source_audio': metadata['source_audio'],
            'status': 'skipped',
            'reason': 'no_stem_candidate_passed_final_validation',

            'min_active_ratio': MIN_ACTIVE_RATIO,
            'max_onset_chroma_difference': MAX_ONSET_CHROMA_DIFFERENCE,
            'onset_threshold': MIN_ONSET_SIMILARITY,
            'pitch_class_span_threshold': MIN_PITCH_CLASS_SPAN,
            'mean_centered_similarity_threshold':
                MIN_MEAN_CENTERED_SIMILARITY,
            'onset_variation_threshold': MIN_ONSET_VARIATION,
            'next_bar_score_gain_threshold': MIN_NEXT_BAR_SCORE_GAIN,

            'bar_alignment_score_keys':
                list(BAR_ALIGNMENT_SCORE_KEYS),

            'similarity_mode': SIMILARITY_MODE,

            'candidate_count': len(metadata['candidates']),
            'eligible_candidate_count': len(eligible),

            'alignment_decisions': alignment_decisions,
            'selected_by_stem_before_final_check':
                selected_by_stem_before_final_check,
            'final_validation': final_validation,

            'selections': {},
        }

        (metadata_path.parent / 'motif_selections.json').write_text(
            json.dumps(
                skipped_metadata,
                ensure_ascii=False,
                indent=2,
            ),
            encoding='utf-8',
        )

        print_motif_result(
            metadata['title'],
            False,
            'stem별 시작점 결정 후 최종 검증 통과 후보 없음',
        )

        return skipped_metadata

    # =========================================================
    # 6. 최종 검증까지 통과한 stem 후보들 중
    #    가장 이른 start_sec 선택
    # =========================================================
    earliest = dict(
        min(
            final_valid_by_stem.values(),
            key=lambda row: row['start_sec'],
        )
    )

    selection_file = 'selected_earliest.flac'

    shutil.copy2(
        metadata_path.parent / earliest['candidate_file'],
        metadata_path.parent / selection_file,
    )

    earliest['selection_file'] = selection_file

    # =========================================================
    # 7. selection metadata 저장
    # =========================================================
    selection_metadata = {
        'song_id': metadata['song_id'],
        'title': metadata['title'],
        'source_audio': metadata['source_audio'],
        'status': 'selected',

        'min_active_ratio': MIN_ACTIVE_RATIO,
        'max_onset_chroma_difference': MAX_ONSET_CHROMA_DIFFERENCE,
        'onset_threshold': MIN_ONSET_SIMILARITY,
        'pitch_class_span_threshold': MIN_PITCH_CLASS_SPAN,
        'mean_centered_similarity_threshold':
            MIN_MEAN_CENTERED_SIMILARITY,
        'onset_variation_threshold': MIN_ONSET_VARIATION,

        'next_bar_score_gain_threshold': MIN_NEXT_BAR_SCORE_GAIN,
        'bar_alignment_score_keys': list(BAR_ALIGNMENT_SCORE_KEYS),

        'similarity_mode': SIMILARITY_MODE,

        'candidate_count': len(metadata['candidates']),
        'eligible_candidate_count': len(eligible),

        # 시작점 보정 과정
        'alignment_decisions': alignment_decisions,

        # 최종 검증 전 stem별 선택 결과
        'selected_by_stem_before_final_check':
            selected_by_stem_before_final_check,

        # 최종 검증 결과
        'final_validation': final_validation,

        # 최종 검증까지 통과한 stem들
        'final_valid_by_stem': final_valid_by_stem,

        # 곡 전체 최종 선택
        'selections': {
            'earliest': earliest,
        },
    }

    (metadata_path.parent / 'motif_selections.json').write_text(
        json.dumps(
            selection_metadata,
            ensure_ascii=False,
            indent=2,
        ),
        encoding='utf-8',
    )

    print_motif_result(
        metadata['title'],
        True,
        (
            f"{earliest['stem_name']} "
            f"@ {earliest['start_sec']:.2f}s"
        ),
    )

    return selection_metadata

all_rows = []
dataset_results = []
audio_failures = []
pending_jobs = []

def is_audio_decode_error(error):
    decode_markers = ('Failed to decode audio samples', 'Could not push packet', 'Invalid data found')
    seen = set()
    while error is not None and id(error) not in seen:
        if any(marker in str(error) for marker in decode_markers):
            return True
        seen.add(id(error))
        error = error.__cause__ or error.__context__
    return False

assert DEMUCS_BATCH_SIZE >= 1
assert MOTIF_WORKERS >= 1
for song_index, audio_path in enumerate(song_paths):
    song = songs[song_index]
    display_title = song_display_title(song)
    print(f'[{song_index + 1}/{len(song_paths)}] {display_title}')
    song_id = diagnostic_song_id(song)
    song_dir = DIAGNOSTIC_DIR / song_id
    if not FORCE_REPROCESS and not song_dir.exists():
        legacy_song_dir = DIAGNOSTIC_DIR / f'{song_index:03d}_{safe_folder_name(display_title)}'
        if legacy_song_dir.exists():
            legacy_song_dir.rename(song_dir)
            print(f'  [rename] {legacy_song_dir.name} -> {song_dir.name}')
    metadata_path = song_dir / 'motif_scores.json'

    existing = None
    if not FORCE_REPROCESS and metadata_path.is_file() and (song_dir / 'melodic_accompaniment.flac').is_file():
        try:
            candidate_metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
            same_audio = (
                Path(candidate_metadata.get('source_audio', '')).resolve()
                == Path(audio_path).resolve()
            )
            same_similarity_mode = candidate_metadata.get('similarity_mode') == SIMILARITY_MODE
            candidates = candidate_metadata.get('candidates')
            candidates_complete = isinstance(candidates, list) and all(
                'pitch_class_span' in row
                and 'mean_centered_onset_similarity' in row
                and 'mean_centered_chroma_similarity' in row
                and (
                    not candidate_passes_export_filters(row)
                    or (
                        row.get('candidate_file')
                        and (song_dir / row['candidate_file']).is_file()
                    )
                )
                for row in candidate_metadata.get('candidates', [])
            )
            if same_audio and same_similarity_mode and candidates_complete:
                existing = candidate_metadata
        except (OSError, ValueError, TypeError):
            existing = None

    if existing is not None:
        selection_path = song_dir / 'motif_selections.json'
        try:
            cached_selection = json.loads(selection_path.read_text(encoding='utf-8'))
        except (OSError, ValueError, TypeError):
            cached_selection = None
        if cached_selection is not None and cached_selection.get('status') == 'skipped':
            same_skip_audio = (
                Path(cached_selection.get('source_audio', '')).resolve()
                == Path(audio_path).resolve()
            )
            same_skip_config = all((
                cached_selection.get('min_active_ratio') == MIN_ACTIVE_RATIO,
                cached_selection.get('max_onset_chroma_difference') == MAX_ONSET_CHROMA_DIFFERENCE,
                cached_selection.get('onset_threshold') == MIN_ONSET_SIMILARITY,
                cached_selection.get('pitch_class_span_threshold') == MIN_PITCH_CLASS_SPAN,
                cached_selection.get('mean_centered_similarity_threshold') == MIN_MEAN_CENTERED_SIMILARITY,
                cached_selection.get('onset_variation_threshold') == MIN_ONSET_VARIATION,
                cached_selection.get('similarity_mode') == SIMILARITY_MODE,
            ))
            if same_skip_audio and same_skip_config:
                for row in existing['candidates']:
                    all_rows.append({'song_id': song_id, 'title': display_title, **row})
                if dataset_builder is not None:
                    dataset_results.append(dataset_builder.record_rejection(
                        song['track'], cached_selection.get('reason', 'final_motif_rejected')
                    ))
                print_motif_result(
                    display_title, False, 'cached: 이전 실행에서 통과한 모티프 없음'
                )
                continue
            print('  [reprocess] 스킵 이후 음원 또는 임계값 변경')
            existing = None

    if existing is not None:
        selection_metadata = save_earliest_valid_selection(metadata_path, existing)
        if dataset_builder is not None:
            if selection_metadata['status'] == 'selected':
                validated_variation = selection_metadata['selections']['earliest']['onset_variation']
                dataset_result = dataset_builder.process_track(
                    song['track'], audio_index,
                    validated_onset_variation=validated_variation,
                )
            else:
                dataset_result = dataset_builder.record_rejection(
                    song['track'], selection_metadata.get('reason', 'final_motif_rejected')
                )
            dataset_results.append(dataset_result)
        elif not song_stems_complete(song, audio_path):
            print('  [process] 저장되지 않은 songs stem 분리')
            stems, sample_rate, mixture = separator.separate(audio_path)
            save_song_stems(song, audio_path, stems, sample_rate)
            del stems, mixture
        print('  [reuse] 기존 분리/후보 결과 사용')
        for row in existing['candidates']:
            all_rows.append({'song_id': song_id, 'title': display_title, **row})
        continue

    pending_jobs.append({
        'song_index': song_index, 'song': song, 'audio_path': audio_path,
        'display_title': display_title, 'song_id': song_id,
        'song_dir': song_dir, 'metadata_path': metadata_path,
    })

for batch_start in range(0, len(pending_jobs), DEMUCS_BATCH_SIZE):
    batch_jobs = pending_jobs[batch_start:batch_start + DEMUCS_BATCH_SIZE]
    batch_paths = [job['audio_path'] for job in batch_jobs]
    batch_number = batch_start // DEMUCS_BATCH_SIZE + 1
    batch_count = (len(pending_jobs) + DEMUCS_BATCH_SIZE - 1) // DEMUCS_BATCH_SIZE
    print(f'[Demucs batch {batch_number}/{batch_count}] {len(batch_jobs)}곡 분리')
    try:
        batch_separations = separator.separate_many(batch_paths)
    except RuntimeError as batch_error:
        if not is_audio_decode_error(batch_error):
            raise

        valid_jobs = []
        for job in batch_jobs:
            try:
                separator._load_audio(job['audio_path'])
            except Exception as audio_error:
                if not is_audio_decode_error(audio_error):
                    raise
                failure = {
                    'song_id': job['song_id'],
                    'title': job['display_title'],
                    'source_audio': str(job['audio_path']),
                    'status': 'failed',
                    'stage': 'audio_decode',
                    'error_type': type(audio_error).__name__,
                    'error_message': str(audio_error),
                }
                audio_failures.append(failure)
                print(f"  [skip: audio decode failed] {job['display_title']}: {audio_error}")
                if dataset_builder is not None:
                    dataset_results.append(dataset_builder.record_rejection(
                        job['song']['track'],
                        f"audio_decode_failed: {type(audio_error).__name__}: {audio_error}",
                    ))
            else:
                valid_jobs.append(job)

        batch_jobs = valid_jobs
        batch_paths = [job['audio_path'] for job in batch_jobs]
        batch_separations = separator.separate_many(batch_paths) if batch_paths else []
    assert len(batch_separations) == len(batch_jobs)
    batch_scores = motif_scorer.score_many(
        [
            (job['audio_path'], separation_result[0], separation_result[1])
            for job, separation_result in zip(batch_jobs, batch_separations)
        ],
        max_workers=MOTIF_WORKERS,
    )

    for job, separation_result, result in zip(batch_jobs, batch_separations, batch_scores):
        song = job['song']
        audio_path = job['audio_path']
        display_title = job['display_title']
        song_id = job['song_id']
        song_dir = job['song_dir']
        metadata_path = job['metadata_path']
        song_dir.mkdir(parents=True, exist_ok=True)
        stems, sample_rate, mixture = separation_result
        print(f'  [motif] {display_title}')
        if dataset_builder is None:
            save_song_stems(song, audio_path, stems, sample_rate)
        melodic = result['melodic_accompaniment']
        save_audio(song_dir / 'melodic_accompaniment.flac', melodic, sample_rate, audio_format='flac')

        for stale_candidate in song_dir.glob('candidate_*.flac'):
            stale_candidate.unlink()
        exported_candidates = [
            row for row in result['candidates'] if candidate_passes_export_filters(row)
        ]
        for row_index, row in enumerate(exported_candidates):
            start = round(row['start_sec'] * sample_rate)
            end = round(row['end_sec'] * sample_rate)
            filename = f"candidate_{row_index:03d}_{row['stem_name']}.flac"
            save_audio(song_dir / filename, melodic[:, start:end], sample_rate, audio_format='flac')
            row['candidate_file'] = filename
        for row in result['candidates']:
            all_rows.append({'song_id': song_id, 'title': display_title, **row})

        metadata = {
            'song_id': song_id, 'title': display_title, 'source_audio': str(audio_path),
            'sample_rate': sample_rate, 'similarity_mode': SIMILARITY_MODE,
            'candidates': result['candidates'],
        }
        metadata_path.write_text(
            json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        selection_metadata = save_earliest_valid_selection(metadata_path, metadata)
        if dataset_builder is not None:
            if selection_metadata['status'] == 'selected':
                validated_variation = selection_metadata['selections']['earliest']['onset_variation']
                dataset_result = dataset_builder.process_track(
                    song['track'],
                    audio_index,
                    separation_result=separation_result,
                    motif_scores=result,
                    validated_onset_variation=validated_variation,
                )
            else:
                dataset_result = dataset_builder.record_rejection(
                    song['track'], selection_metadata.get('reason', 'final_motif_rejected')
                )
            dataset_results.append(dataset_result)
        del stems, mixture, result, melodic

failure_report_path = DIAGNOSTIC_DIR / 'audio_failures.jsonl'
failure_report_path.write_text(
    ''.join(json.dumps(failure, ensure_ascii=False) + '\n' for failure in audio_failures),
    encoding='utf-8',
)
if audio_failures:
    print(f'[audio failures] {len(audio_failures)}곡: {failure_report_path}')

scores_df = pd.DataFrame(all_rows)
scores_df.to_csv(DIAGNOSTIC_DIR / 'all_motif_scores.csv', index=False)
if scores_df.empty:
    display(scores_df)
else:
    display(scores_df.sort_values(['song_id', 'start_sec', 'stem_name']))
if dataset_results:
    from dataclasses import asdict
    display(pd.DataFrame([asdict(result) for result in dataset_results]))

accepted_track_ids = sorted({
    result.track_id
    for result in dataset_results
    if result.status in {'accepted', 'skipped'} and result.output_dir is not None
})
motif_version = {
    'schema': 'mohim_motif_dataset_v5',
    'data_source': DATA_SOURCE,
    'max_songs': MAX_SONGS,
    'separator_model': 'htdemucs_6s',
    'demucs_shifts': DEMUCS_SHIFTS,
    'bars': MOTIF_BARS,
    'search_seconds': MOTIF_SEARCH_SECONDS,
    'min_active_ratio': MIN_ACTIVE_RATIO,
    'max_onset_chroma_difference': MAX_ONSET_CHROMA_DIFFERENCE,
    'min_onset_similarity': MIN_ONSET_SIMILARITY,
    'min_pitch_class_span': MIN_PITCH_CLASS_SPAN,
    'min_mean_centered_similarity': MIN_MEAN_CENTERED_SIMILARITY,
    'min_onset_variation': MIN_ONSET_VARIATION,
    'similarity_mode': SIMILARITY_MODE,
    'accepted_track_ids': accepted_track_ids,
}
MOTIF_VERSION_PATH.write_text(
    json.dumps(motif_version, ensure_ascii=False, sort_keys=True, indent=2) + '\n',
    encoding='utf-8',
)
print('motif dataset version:', MOTIF_VERSION_PATH)

[1/55] Doja Cat - Agora Hills
[2/55] Doja Cat - Need To Know
[3/55] Jack Harlow - Lovin On Me
[4/55] Justin Bieber - Yukon
[5/55] SHAED - Trampoline
[6/55] Ed Sheeran - Shape Of You
[7/55] Olivia Rodrigo - Good 4 U
[8/55] Post Malone - Better Now
[9/55] Shawn Mendes - Stitches
[10/55] Alessia Cara - Scars To Your Beautiful
[11/55] Jonas Brothers - Sucker
[12/55] Lizzo - Truth Hurts
[13/55] LMFAO Featuring Lauren Bennett & GoonRock - Party Rock Anthem
[14/55] Panic! At The Disco - High Hopes
[15/55] Kehlani - Folded
[16/55] Dua Lipa - Break My Heart
[17/55] Ellie Goulding - Lights
[18/55] Justin Bieber - Daisies
[19/55] Khalid - Talk
[20/55] Olivia Dean - So Easy (To Fall In Love)
[21/55] Sabrina Carpenter - Taste
[22/55] SZA - Kill Bill
[23/55] The All-American Rejects - Gives You Hell
[24/55] DNCE - Cake By The Ocean
[25/55] Camila Cabello - Never Be The Same
[26/55] Charlie Puth - Attention
[27/55] Trevor Daniel - Falling
[28/55] Benson Boone - Sorry I'm Here For Someone Else
[29/55]

## 4. 현재 곡의 highest similarity motif 일괄 재생

현재 입력에 포함된 곡 중 저장된 highest similarity motif를 `DEBUG_TRACK_INDEX`부터 `DEBUG_TRACK_COUNT`곡씩 표시합니다. 예: 11~20곡은 `DEBUG_TRACK_INDEX = 10`으로 설정합니다.